# Multilingual GoEmotions Dataset
Now that we are going to move on to testing the efficacy of our model when looking at a language other than English, we needed to find another dataset that includes another language. 

The Multilingual GoEmotions dataset is an extension of the GoEmotions dataset that we previously used and includes the same labels/structure as that dataset too. The languages included in the dataset are Arabic, French, Spanish, Dutch, and Turkish in addition to the original English dataset. 

The language that we'll be working with specifically is the French subset of the Multilingual GoEmotions Dataset. 

The dataset can be found on Hugging Face here: https://huggingface.co/datasets/AnasAlokla/multilingual_go_emotions

<iframe
  src="https://huggingface.co/datasets/AnasAlokla/multilingual_go_emotions/embed/viewer/default/train"
  frameborder="0"
  width="100%"
  height="560px"
></iframe>


In [1]:
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
splits = {'train': 'train_dataset.csv', 'validation': 'validation_dataset.csv', 'test': 'test_dataset.csv'}
train_raw = pd.read_csv("hf://datasets/AnasAlokla/multilingual_go_emotions/" + splits["train"])
validation_raw = pd.read_csv("hf://datasets/AnasAlokla/multilingual_go_emotions/" + splits["validation"])
test_raw = pd.read_csv("hf://datasets/AnasAlokla/multilingual_go_emotions/" + splits["test"])

C:\Users\nsdiz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Filtering to include only French comments
train_f = train_raw[train_raw["language"] == "fr"]
validation_f = validation_raw[validation_raw["language"] == "fr"]
test_f = test_raw[test_raw["language"] == "fr"]

In [3]:
# Check for empty values, null values, and duplicates
print(f"Train null text fields: {train_f["text"].isnull().sum()}")
print(f"Train empty text fields: {(train_f['text'].str.strip() == '').sum()}")
print(f"Train duplicates: {train_f.duplicated().sum()}\n")

print(f"Validation null text fields: {validation_f["text"].isnull().sum()}")
print(f"Validation empty text fields: {(validation_f['text'].str.strip() == '').sum()}")
print(f"Validation duplicates: {validation_f.duplicated().sum()}\n")

print(f"Test null text fields train: {test_f["text"].isnull().sum()}")
print(f"Test empty text fields: {(test_f['text'].str.strip() == '').sum()}")
print(f"Test duplicates: {test_f.duplicated().sum()}")

Train null text fields: 2
Train empty text fields: 0
Train duplicates: 0

Validation null text fields: 2
Validation empty text fields: 0
Validation duplicates: 0

Test null text fields train: 1
Test empty text fields: 0
Test duplicates: 0


In [4]:
# Removing null text fields from the datasets
train_processed = train_f[train_f["text"].isnull() == False]
validation_processed = validation_f[validation_f["text"].isnull() == False]
test_processed = test_f[test_f["text"].isnull() == False]

In [5]:
# Removing placeholder words, links, and extra whitespace from text
import re

def clean_text(text):
  text = re.sub(r"\[NOM\]|\[NAME\]|\[RELIGION\]", "", text) #remove hidden placeholder words
  text = re.sub(r"http\S+", "", text)               #remove links
  text = re.sub(r"\s+", " ", text).strip()          #remove extra whitespace   
  return text

train_processed["clean_text"] = train_processed["text"].apply(clean_text)
validation_processed["clean_text"] = validation_processed["text"].apply(clean_text)
test_processed["clean_text"] = test_processed["text"].apply(clean_text)

# Removing rows that who's clean text has become empty after removing placeholder words
train_processed = train_processed[train_processed["clean_text"].str.strip() != ""]
validation_processed = validation_processed[validation_processed["clean_text"].str.strip() != ""]
test_processed = test_processed[test_processed["clean_text"].str.strip() != ""]

In [6]:
# Categorizing data points into positive, negative, neutral, ambiguous from their existing labels
import ast

LABEL_NAMES = [
  "admiration","amusement","anger","annoyance","approval","caring",
  "confusion","curiosity","desire","disappointment","disapproval",
  "disgust","embarrassment","excitement","fear","gratitude","grief",
  "joy","love","nervousness","optimism","pride","realization","relief",
  "remorse","sadness","surprise","neutral"
]
# Map each label to its corresponding group
GROUP_MAP = {
  "admiration":"positive","amusement":"positive","approval":"positive",
  "caring":"positive","desire":"positive","excitement":"positive",
  "gratitude":"positive","joy":"positive","love":"positive",
  "optimism":"positive","pride":"positive","relief":"positive",
  "anger":"negative","annoyance":"negative","disappointment":"negative",
  "disapproval":"negative","disgust":"negative","embarrassment":"negative",
  "fear":"negative","grief":"negative","nervousness":"negative",
  "remorse":"negative","sadness":"negative",
  "confusion":"ambiguous","curiosity":"ambiguous",
  "realization":"ambiguous","surprise":"ambiguous",
  "neutral":"neutral"
}
# Function to convert label IDs to their corresponding groups
def labels_to_groups_all(label_ids):
  names = [LABEL_NAMES[i] for i in label_ids]
  groups = [GROUP_MAP[n] for n in names]
  return list(dict.fromkeys(groups))

# Process each dataset split
def process_split(df):
  df = df.copy()
  df["all_groups"] = df["labels"].apply(labels_to_groups_all)
  df["num_distinct_groups"] = df["all_groups"].apply(len)
  clean = df[df["num_distinct_groups"] == 1].copy()
  clean["group_label"] = clean["all_groups"].apply(lambda g: g[0])
  conflict = df[df["num_distinct_groups"] > 1].copy()
  return clean, conflict

# Converting the strings in the labels column into the arrays they represent
train_processed['labels'] = train_processed['labels'].apply(ast.literal_eval)
validation_processed['labels'] = validation_processed['labels'].apply(ast.literal_eval)
test_processed['labels'] = test_processed['labels'].apply(ast.literal_eval)

# Process all dataset splits
train_clean_f, train_conflict = process_split(train_processed)
val_clean_f, val_conflict = process_split(validation_processed)
test_clean_f, test_conflict = process_split(test_processed)

In [7]:
# Viewing the category distribution of each data set
print(train_clean_f["group_label"].value_counts())
print(val_clean_f["group_label"].value_counts())
print(test_clean_f["group_label"].value_counts())

group_label
positive     15255
neutral      12790
negative      8160
ambiguous     3827
Name: count, dtype: int64
group_label
positive     1868
neutral      1645
negative     1042
ambiguous     477
Name: count, dtype: int64
group_label
positive     1896
neutral      1577
negative     1013
ambiguous     501
Name: count, dtype: int64
